# 📊 Web Scraping Veri Analizi ve Görselleştirme

Bu notebook'ta, scraping yoluyla elde ettiğimiz verileri analiz edeceğiz. 

**Hedefler:**
1. Veriyi yükleme (JSON veya MongoDB)
2. Temel istatistikler
3. Görselleştirme (Grafikler)
4. Kelime Bulutu (WordCloud)

In [ ]:
# Gerekli kütüphaneleri import edelim
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from wordcloud import WordCloud

# Grafik ayarları
plt.style.use('ggplot')
sns.set_palette("husl")

## 1. Veriyi Yükleme

Veriyi daha önce kaydettiğimiz JSON dosyasından veya MongoDB'den çekebiliriz. 
Bu örnekte kolaylık olması için **Module 02**'de ürettiğimiz JSON dosyasını kullanacağız.

In [ ]:
# Veri kaynağını belirle (Dosya yolunu kendi sistemine göre düzenle)
# Not: Eğer 02-bs4-requests klasöründeki dosyayı kullanacaksak:
json_path = '../../02-bs4-requests/scraped_data/quotes_structured.json'

# Eğer dosya yoksa dummy data oluşturalım (Hata almamak için)
if not os.path.exists(json_path):
    print("⚠️ Dosya bulunamadı, örnek veri kullanılıyor.")
    data = [
        {"text": "Quote 1", "author": "Einstein", "tags": ["science", "life"]},
        {"text": "Quote 2", "author": "Einstein", "tags": ["physics"]},
        {"text": "Quote 3", "author": "Rowling", "tags": ["magic"]}
    ]
    df = pd.DataFrame(data)
else:
    # JSON dosyasını yükle
    with open(json_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
    
    # quotes_structured.json yapısında veriler 'quotes' anahtarı altındaydı
    if 'quotes' in raw_data:
        df = pd.DataFrame(raw_data['quotes'])
    else:
        df = pd.DataFrame(raw_data)

print(f"✅ Veri yüklendi. Toplam kayıt: {len(df)}")
df.head()

## 2. Temel Analizler

In [ ]:
# En çok alıntısı olan yazarlar
top_authors = df['author'].value_counts().head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_authors.values, y=top_authors.index)
plt.title('En Çok Alıntısı Olan Yazarlar')
plt.xlabel('Alıntı Sayısı')
plt.show()

In [ ]:
# Alıntı uzunluk analizi (Kelime sayısı)
df['word_count'] = df['text'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 6))
sns.histplot(df['word_count'], bins=20, kde=True)
plt.title('Alıntı Uzunluk Dağılımı (Kelime Sayısı)')
plt.xlabel('Kelime Sayısı')
plt.ylabel('Frekans')
plt.show()

## 3. Etiket (Tag) Analizi
Etiketler liste içinde liste olduğu için (nested list), önce bunları düzleştirmemiz (explode) gerekir.

In [ ]:
# Tag'leri düzleştir
all_tags = df.explode('tags')

# En popüler 15 etiket
top_tags = all_tags['tags'].value_counts().head(15)

plt.figure(figsize=(14, 6))
sns.barplot(x=top_tags.index, y=top_tags.values, palette="viridis")
plt.title('En Popüler Konular (Tags)')
plt.xticks(rotation=45)
plt.show()

## 4. Word Cloud (Kelime Bulutu)
Alıntılarda en çok geçen kelimeleri görselleştirelim.

In [ ]:
# Tüm metinleri birleştir
text_corpus = " ".join(quote for quote in df.text)

# WordCloud oluştur
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text_corpus)

plt.figure(figsize=(15, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis("off")
plt.title("Quotes Word Cloud")
plt.show()